<a href="https://colab.research.google.com/github/AK-SINGH-0/AK-SINGH-0/blob/main/fedshield_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r "/content/drive/MyDrive/FL_IDS_Research/CIC_IOT_2023.csv" /content/

In [ ]:
!pip install shap torch torchvision scikit-learn pandas numpy matplotlib seaborn tqdm -q
print("✅ All libraries installed!")




In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import shap
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import (LabelEncoder, StandardScaler,
                                    label_binarize)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, f1_score, accuracy_score,
    precision_score, recall_score
)
from copy import deepcopy
from collections import defaultdict
import warnings, random, os, time, zipfile, io
warnings.filterwarnings('ignore')

# ── Reproducibility ────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# ── Device ─────────────────────────────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Hyperparameters (edit here if needed) ─────────────────────────
NUM_CLIENTS       = 5        # Simulated IoT device types
FL_ROUNDS         = 20       # Federated Learning rounds
LOCAL_EPOCHS      = 5        # Local epochs per FL round
BATCH_SIZE        = 256
LEARNING_RATE     = 0.001
MU                = 0.01     # FedProx proximal term (key for Non-IID)
DIRICHLET_ALPHA   = 0.5      # Non-IID severity (lower = more skewed)
FRACTION_CLIENTS  = 1.0      # Fraction of clients selected per round
DROPOUT           = 0.3      # Model dropout rate
SHAP_BACKGROUND   = 100      # Background samples for SHAP
SHAP_EXPLAIN      = 200      # Samples to explain with SHAP

print(f"✅ Configuration loaded")
print(f"   Device      : {DEVICE}")
print(f"   FL Rounds   : {FL_ROUNDS} | Clients: {NUM_CLIENTS}")
print(f"   FedProx μ   : {MU}  | Dirichlet α: {DIRICHLET_ALPHA}")

In [ ]:
from google.colab import files as colab_files

print("📂 Please upload your CIC_IoT2023.csv file...")
uploaded = colab_files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_csv(io.BytesIO(uploaded[filename]))
print(f"\n✅ Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\n── Column preview ──────────────────────────────────")
print(df.dtypes.to_string())

# ── Auto-detect label column ───────────────────────────────────────
LABEL_CANDIDATES = [
    'label', 'Label', 'Attack_type', 'attack_type',
    'Class', 'class', 'Category', 'category', 'Target'
]
label_col = next(
    (c for c in LABEL_CANDIDATES if c in df.columns),
    df.columns[-1]           # fallback: last column
)
print(f"\n🏷️  Label column detected : '{label_col}'")
print(f"\n── Class distribution ──────────────────────────────")
print(df[label_col].value_counts().to_string())

📂 Please upload your CIC_IoT2023.csv file...


In [ ]:
print("\n🔧 Preprocessing data...")

# 1. Drop exact duplicates
before = len(df)
df.drop_duplicates(inplace=True)
print(f"  ✔ Duplicates removed: {before - len(df):,}")

# 2. Separate features and label
X_raw = df.drop(columns=[label_col])
y_raw = df[label_col].copy()

# 3. Keep only numeric features
X_raw = X_raw.select_dtypes(include=[np.number])

# 4. Replace ±inf with NaN, then fill NaN with column median
X_raw.replace([np.inf, -np.inf], np.nan, inplace=True)
X_raw.fillna(X_raw.median(), inplace=True)

# 5. Remove zero-variance columns (carry no information)
zero_var = X_raw.columns[X_raw.var() == 0].tolist()
X_raw.drop(columns=zero_var, inplace=True)
print(f"  ✔ Zero-variance columns removed: {len(zero_var)}")
print(f"  ✔ Final feature count: {X_raw.shape[1]}")

FEATURE_NAMES = list(X_raw.columns)

# 6. Encode string labels → integers
le = LabelEncoder()
y_encoded = le.fit_transform(y_raw)
CLASS_NAMES = list(le.classes_)
NUM_CLASSES = len(CLASS_NAMES)
print(f"  ✔ Classes ({NUM_CLASSES}): {CLASS_NAMES}")

# 7. Stratified train / test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_raw.values, y_encoded,
    test_size=0.2, random_state=SEED, stratify=y_encoded
)

# 8. Standardise features (fit only on train, apply to test)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

INPUT_DIM = X_train.shape[1]
print(f"  ✔ Train: {X_train.shape} | Test: {X_test.shape}")
print("✅ Preprocessing complete!")


In [ ]:
def dirichlet_partition(X, y, num_clients, alpha, seed=SEED):
    """
    Partition training data into Non-IID subsets per client.

    Uses a Dirichlet distribution so each 'IoT device' sees a
    different skew of attack classes — exactly like the real world
    where a smart camera faces DDoS floods but a thermostat sees
    mostly benign traffic.

    alpha=0.5  → strong Non-IID (realistic)
    alpha=100  → nearly IID (ideal / unrealistic)
    """
    np.random.seed(seed)
    num_classes = len(np.unique(y))
    client_data = {i: {'X': [], 'y': []} for i in range(num_clients)}

    for cls in range(num_classes):
        cls_idx = np.where(y == cls)[0]
        np.random.shuffle(cls_idx)

        proportions = np.random.dirichlet(alpha * np.ones(num_clients))
        proportions = (proportions * len(cls_idx)).astype(int)
        proportions[-1] += len(cls_idx) - proportions.sum()  # fix rounding

        start = 0
        for cid, count in enumerate(proportions):
            end = start + count
            client_data[cid]['X'].append(X[cls_idx[start:end]])
            client_data[cid]['y'].append(y[cls_idx[start:end]])
            start = end

    for cid in range(num_clients):
        client_data[cid]['X'] = np.vstack(client_data[cid]['X'])
        client_data[cid]['y'] = np.concatenate(client_data[cid]['y'])

    return client_data


client_data = dirichlet_partition(X_train, y_train, NUM_CLIENTS, DIRICHLET_ALPHA)

# ── Visualise Non-IID distribution across clients ─────────────────
fig, axes = plt.subplots(1, NUM_CLIENTS, figsize=(4 * NUM_CLIENTS, 4),
                          facecolor='#0d0d1a')
colors = plt.cm.plasma(np.linspace(0.1, 0.9, NUM_CLASSES))

for cid in range(NUM_CLIENTS):
    counts = np.bincount(client_data[cid]['y'], minlength=NUM_CLASSES)
    ax = axes[cid]
    ax.set_facecolor('#0d0d1a')
    bars = ax.bar(range(NUM_CLASSES), counts, color=colors, edgecolor='none',
                  width=0.8)
    ax.set_title(f'Client {cid + 1}\n({len(client_data[cid]["y"]):,} samples)',
                 fontsize=10, color='white', fontweight='bold')
    ax.set_xlabel('Class Index', fontsize=8, color='#aaaaaa')
    ax.tick_params(colors='#aaaaaa', labelsize=7)
    ax.spines[:].set_visible(False)
    if cid == 0:
        ax.set_ylabel('Sample Count', fontsize=8, color='#aaaaaa')

fig.suptitle(
    f'Non-IID Data Distribution Across IoT Clients  (Dirichlet α={DIRICHLET_ALPHA})',
    fontsize=14, fontweight='bold', color='white', y=1.03
)
plt.tight_layout()
plt.savefig('01_non_iid_distribution.png', dpi=150, bbox_inches='tight',
            facecolor='#0d0d1a')
plt.show()

for cid in range(NUM_CLIENTS):
    counts = np.bincount(client_data[cid]['y'], minlength=NUM_CLASSES)
    dominant = CLASS_NAMES[np.argmax(counts)]
    print(f"  Client {cid+1}: {len(client_data[cid]['y']):,} samples | "
          f"dominant class → '{dominant}'")

print("\n✅ Non-IID partitions created!")

In [ ]:
class IDSNet(nn.Module):
    """
    Feed-forward Deep Neural Network for IoT Intrusion Detection.

    Architecture:
        Input(D) → [Linear→BN→ReLU→Dropout] × 3 → Linear → Softmax

    Chosen because:
    • Tabular network data suits DNN better than CNN/LSTM
    • BatchNorm stabilises Non-IID federated training
    • Dropout prevents overfitting on small client datasets
    """
    def __init__(self, input_dim, num_classes, dropout=DROPOUT):
        super(IDSNet, self).__init__()
        self.net = nn.Sequential(
            # Block 1
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            # Block 2
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            # Block 3
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            # Output
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        return self.net(x)


# Sanity-check the model
_test_model = IDSNet(INPUT_DIM, NUM_CLASSES).to(DEVICE)
_params = sum(p.numel() for p in _test_model.parameters())
print(f"✅ IDSNet built")
print(f"   Input dim  : {INPUT_DIM}")
print(f"   Num classes: {NUM_CLASSES}")
print(f"   Parameters : {_params:,}")
del _test_model

In [ ]:
def make_loader(X, y, batch_size=BATCH_SIZE, shuffle=True):
    """Wrap numpy arrays in a PyTorch DataLoader."""
    ds = TensorDataset(torch.FloatTensor(X), torch.LongTensor(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      drop_last=False)


def fedprox_local_train(model, global_model, data, epochs,
                         lr=LEARNING_RATE, mu=MU, device=DEVICE):
    """
    FedProx local training.

    Loss = CrossEntropy  +  (μ/2) × ||w − w_global||²
                                    ↑
                            Proximal term that penalises the local
                            model for drifting far from the global
                            model.  This is the key fix for Non-IID.

    Without this term (plain FedAvg), divergent local updates cause
    the global model to 'forget' minority attack classes.
    """
    model.train()
    loader    = make_loader(data['X'], data['y'])
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    # Cache global weights (no gradient needed)
    global_params = {n: p.detach().clone().to(device)
                     for n, p in global_model.named_parameters()}

    epoch_loss = 0.0
    for _ in range(epochs):
        for X_b, y_b in loader:
            X_b, y_b = X_b.to(device), y_b.to(device)
            optimizer.zero_grad()

            # Standard classification loss
            outputs = model(X_b)
            ce_loss = criterion(outputs, y_b)

            # FedProx proximal regularisation
            prox = sum(
                ((p - global_params[n]) ** 2).sum()
                for n, p in model.named_parameters()
            )

            loss = ce_loss + (mu / 2.0) * prox
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            epoch_loss += loss.item()

    return model, epoch_loss


def fedavg_aggregate(global_model, client_models, client_sizes):
    """
    Federated Averaging (McMahan et al., 2017).
    Weighted average of client model parameters by data volume.
    """
    total = sum(client_sizes)
    g_dict = global_model.state_dict()

    for key in g_dict.keys():
        g_dict[key] = torch.zeros_like(g_dict[key], dtype=torch.float32)
        for m, sz in zip(client_models, client_sizes):
            g_dict[key] += (sz / total) * m.state_dict()[key].float()

    global_model.load_state_dict(g_dict)
    return global_model


def evaluate(model, X, y, device=DEVICE):
    """Return accuracy, F1, precision, recall, predictions, true labels, probs."""
    model.eval()
    loader = make_loader(X, y, shuffle=False)
    preds, labels, probs = [], [], []

    with torch.no_grad():
        for X_b, y_b in loader:
            out  = model(X_b.to(device))
            prob = torch.softmax(out, dim=1).cpu().numpy()
            pred = out.argmax(dim=1).cpu().numpy()
            preds.extend(pred)
            labels.extend(y_b.numpy())
            probs.extend(prob)

    preds  = np.array(preds)
    labels = np.array(labels)
    probs  = np.array(probs)

    acc  = accuracy_score(labels, preds)
    f1   = f1_score(labels, preds, average='weighted', zero_division=0)
    prec = precision_score(labels, preds, average='weighted', zero_division=0)
    rec  = recall_score(labels, preds, average='weighted', zero_division=0)

    return acc, f1, prec, rec, preds, labels, probs


print("✅ Helper functions defined!")

In [ ]:
FL_ROUNDS = 35
LOCAL_EPOCHS = 10

print("″Starting FedProx Federated Learning Training…\n")
print(f"   Clients: {NUM_CLIENTS} | Rounds: {FL_ROUNDS} | "
      f"Local epochs: {LOCAL_EPOCHS}")
print(f"   FedProx μ = {MU}   Dirichlet α = {DIRICHLET_ALPHA}\n")
print("─" * 72)
print(f"{'Round':>6}  {'TrainAcc':>9}  {'TestAcc':>9}  "
      f"{'TrainF1':>9}  {'TestF1':>9}  {'Time(s)':>8}")
print("─" * 72)

# Initialise global model
global_model = IDSNet(INPUT_DIM, NUM_CLASSES).to(DEVICE)

history = defaultdict(list)
start_total = time.time()

for fl_round in range(1, FL_ROUNDS + 1):
    t0 = time.time()

    # Select clients for this round
    n_sel = max(1, int(FRACTION_CLIENTS * NUM_CLIENTS))
    selected = random.sample(range(NUM_CLIENTS), n_sel)

    client_models = []
    client_sizes  = []

    for cid in selected:
        # Each client trains a copy of the global model locally
        local_m = deepcopy(global_model).to(DEVICE)
        local_m, _ = fedprox_local_train(
            local_m, global_model, client_data[cid], LOCAL_EPOCHS
        )
        client_models.append(local_m)
        client_sizes.append(len(client_data[cid]['y']))

    # Aggregate client models → new global model
    global_model = fedavg_aggregate(global_model, client_models, client_sizes)

    # Evaluate global model
    tr_acc, tr_f1, _, _, _, _, _  = evaluate(global_model, X_train, y_train)
    te_acc, te_f1, te_p, te_r, _, _, _ = evaluate(global_model, X_test, y_test)

    elapsed = time.time() - t0
    history['round'].append(fl_round)
    history['train_acc'].append(tr_acc);  history['test_acc'].append(te_acc)
    history['train_f1'].append(tr_f1);   history['test_f1'].append(te_f1)
    history['test_prec'].append(te_p);   history['test_rec'].append(te_r)

    # Mark best round
    best_tag = " ◀ BEST" if te_acc == max(history['test_acc']) else ""
    print(f"{fl_round:>6}  {tr_acc:>9.4f}  {te_acc:>9.4f}  "
          f"{tr_f1:>9.4f}  {te_f1:>9.4f}  {elapsed:>8.1f}{best_tag}")

total_time = time.time() - start_total
print("─" * 72)
print(f"\n✅ Training complete in {total_time / 60:.1f} minutes")
print(f"🏆 Best Test Accuracy : {max(history['test_acc']):.4f} "
      f"({max(history['test_acc']) * 100:.2f}%)")
print(f"🏆 Best Test F1 Score : {max(history['test_f1']):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor='#0d0d1a')

DARK_BG   = '#0d0d1a'
GRID_COL  = '#1e1e3a'
BLUE      = '#4fc3f7'
ORANGE    = '#ff8a65'
ACCENT    = '#a5d6a7'

for ax in axes:
    ax.set_facecolor(DARK_BG)
    ax.spines[:].set_color(GRID_COL)
    ax.tick_params(colors='#aaaaaa')
    ax.yaxis.label.set_color('#dddddd')
    ax.xaxis.label.set_color('#dddddd')

rounds = history['round']

# ── Left: Accuracy ────────────────────────────────────────────────
axes[0].plot(rounds, history['train_acc'], color=BLUE, lw=2.5,
             marker='o', markersize=4, label='Train Accuracy')
axes[0].plot(rounds, history['test_acc'],  color=ORANGE, lw=2.5,
             marker='s', markersize=4, label='Test Accuracy')
best_acc  = max(history['test_acc'])
best_rnd  = history['round'][history['test_acc'].index(best_acc)]
axes[0].axhline(best_acc, color=ACCENT, lw=1.2, ls='--',
                 label=f'Best Test: {best_acc:.4f} (Round {best_rnd})')
axes[0].set_xlabel('Federated Round', fontsize=12)
axes[0].set_ylabel('Accuracy', fontsize=12)
axes[0].set_title('FedProx Convergence — Accuracy',
                   fontsize=13, fontweight='bold', color='white')
axes[0].legend(fontsize=10, facecolor='#1a1a2e', labelcolor='white')
axes[0].set_ylim([0, 1.05])
axes[0].grid(True, color=GRID_COL, lw=0.8)

# ── Right: F1 Score ───────────────────────────────────────────────
axes[1].plot(rounds, history['train_f1'], color=BLUE, lw=2.5,
             marker='o', markersize=4, label='Train F1')
axes[1].plot(rounds, history['test_f1'],  color=ORANGE, lw=2.5,
             marker='s', markersize=4, label='Test F1')
best_f1   = max(history['test_f1'])
axes[1].axhline(best_f1, color=ACCENT, lw=1.2, ls='--',
                 label=f'Best Test F1: {best_f1:.4f}')
axes[1].set_xlabel('Federated Round', fontsize=12)
axes[1].set_ylabel('Weighted F1 Score', fontsize=12)
axes[1].set_title('FedProx Convergence — F1 Score',
                   fontsize=13, fontweight='bold', color='white')
axes[1].legend(fontsize=10, facecolor='#1a1a2e', labelcolor='white')
axes[1].set_ylim([0, 1.05])
axes[1].grid(True, color=GRID_COL, lw=0.8)

fig.suptitle(
    'FedProx Federated Learning on CIC-IoT2023\n'
    f'({NUM_CLIENTS} clients, Dirichlet α={DIRICHLET_ALPHA}, μ={MU})',
    fontsize=14, fontweight='bold', color='white', y=1.03
)
plt.tight_layout()
plt.savefig('02_fl_convergence.png', dpi=150, bbox_inches='tight',
            facecolor=DARK_BG)
plt.show()
print("✅ Convergence plot saved!")

In [ ]:
te_acc, te_f1, te_p, te_r, y_pred, y_true, y_probs = \
    evaluate(global_model, X_test, y_test)

cm     = confusion_matrix(y_true, y_pred)
cm_pct = cm.astype('float') / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(max(14, NUM_CLASSES * 2),
                                         max(6, NUM_CLASSES * 1.1)),
                          facecolor=DARK_BG)

kws = dict(annot=True, xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
           linewidths=0.5, linecolor='#111')

# Raw counts
sns.heatmap(cm, fmt='d', cmap='Blues', ax=axes[0], **kws,
            annot_kws={'size': max(6, 9 - NUM_CLASSES // 4)})
axes[0].set_title('Confusion Matrix (Counts)',
                   fontsize=13, fontweight='bold', color='white', pad=12)
axes[0].set_xlabel('Predicted', fontsize=11, color='#dddddd')
axes[0].set_ylabel('True Label', fontsize=11, color='#dddddd')
axes[0].tick_params(colors='#cccccc', rotation=45, labelsize=8)

# Normalised (%)
sns.heatmap(cm_pct, fmt='.2f', cmap='Greens', ax=axes[1], **kws,
            vmin=0, vmax=1,
            annot_kws={'size': max(6, 9 - NUM_CLASSES // 4)})
axes[1].set_title('Confusion Matrix (Normalised)',
                   fontsize=13, fontweight='bold', color='white', pad=12)
axes[1].set_xlabel('Predicted', fontsize=11, color='#dddddd')
axes[1].set_ylabel('True Label', fontsize=11, color='#dddddd')
axes[1].tick_params(colors='#cccccc', rotation=45, labelsize=8)

fig.suptitle('FedProx IDS — Confusion Matrix on CIC-IoT2023 Test Set',
             fontsize=14, fontweight='bold', color='white', y=1.01)
plt.tight_layout()
plt.savefig('03_confusion_matrix.png', dpi=150, bbox_inches='tight',
            facecolor=DARK_BG)
plt.show()
print("✅ Confusion matrix saved!")


In [ ]:
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES,
                                output_dict=True, zero_division=0)

print("\n📋 Classification Report:")
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES,
                              zero_division=0))

# ── Per-class F1 bar chart ────────────────────────────────────────
class_f1s = [report[c]['f1-score'] for c in CLASS_NAMES]
bar_colors = plt.cm.RdYlGn(np.array(class_f1s))

fig, ax = plt.subplots(figsize=(max(12, len(CLASS_NAMES) * 1.5), 5),
                        facecolor=DARK_BG)
ax.set_facecolor(DARK_BG)
ax.spines[:].set_color(GRID_COL)
ax.tick_params(colors='#aaaaaa')

bars = ax.bar(CLASS_NAMES, class_f1s, color=bar_colors,
               edgecolor='white', linewidth=0.5, width=0.65)

# Value labels above each bar
for bar, val in zip(bars, class_f1s):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.012,
            f'{val:.3f}', ha='center', va='bottom',
            fontsize=9, fontweight='bold', color='white')

wav_f1 = report['weighted avg']['f1-score']
ax.axhline(wav_f1, color=ACCENT, lw=2, ls='--',
            label=f'Weighted Avg F1: {wav_f1:.4f}')
ax.set_xlabel('Attack Category', fontsize=12, color='#dddddd')
ax.set_ylabel('F1 Score', fontsize=12, color='#dddddd')
ax.set_title('Per-Class F1 Score — FedProx IDS on CIC-IoT2023',
              fontsize=13, fontweight='bold', color='white')
ax.set_ylim([0, 1.15])
ax.tick_params(axis='x', rotation=45, labelsize=9)
ax.legend(fontsize=11, facecolor='#1a1a2e', labelcolor='white')
ax.grid(axis='y', color=GRID_COL, lw=0.8)

plt.tight_layout()
plt.savefig('04_per_class_f1.png', dpi=150, bbox_inches='tight',
            facecolor=DARK_BG)
plt.show()
print("✅ Per-class F1 chart saved!")

In [ ]:
y_test_bin = label_binarize(y_true, classes=range(NUM_CLASSES))

fig, ax = plt.subplots(figsize=(10, 7), facecolor=DARK_BG)
ax.set_facecolor(DARK_BG)
ax.spines[:].set_color(GRID_COL)
ax.tick_params(colors='#aaaaaa')

palette = plt.cm.tab20(np.linspace(0, 1, NUM_CLASSES))
roc_aucs = {}

for i, (cls_name, color) in enumerate(zip(CLASS_NAMES, palette)):
    if y_test_bin[:, i].sum() == 0:
        continue
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_probs[:, i])
    roc_auc = auc(fpr, tpr)
    roc_aucs[cls_name] = roc_auc
    ax.plot(fpr, tpr, color=color, lw=2.2,
            label=f'{cls_name}  (AUC={roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'w--', lw=1.2, alpha=0.5, label='Random baseline')
macro_auc = np.mean(list(roc_aucs.values()))
ax.set_xlabel('False Positive Rate', fontsize=12, color='#dddddd')
ax.set_ylabel('True Positive Rate', fontsize=12, color='#dddddd')
ax.set_title(f'ROC Curves (One-vs-Rest) — Macro AUC = {macro_auc:.4f}',
              fontsize=13, fontweight='bold', color='white')
ax.legend(loc='lower right', fontsize=8, facecolor='#1a1a2e',
           labelcolor='white', ncol=2)
ax.grid(color=GRID_COL, lw=0.8)

plt.tight_layout()
plt.savefig('05_roc_curves.png', dpi=150, bbox_inches='tight',
            facecolor=DARK_BG)
plt.show()
print(f"✅ ROC curves saved!  Macro-Avg AUC: {macro_auc:.4f}")

In [ ]:
print("\n🔍 Computing SHAP values…  (may take 3–6 minutes)")
print("   Using DeepExplainer (fast, designed for neural networks)")

global_model.eval()
global_model.cpu()

bg_idx  = np.random.choice(len(X_train), SHAP_BACKGROUND, replace=False)
exp_idx = np.random.choice(len(X_test),  SHAP_EXPLAIN,    replace=False)

X_bg  = torch.FloatTensor(X_train[bg_idx])
X_exp = torch.FloatTensor(X_test[exp_idx])

explainer   = shap.DeepExplainer(global_model, X_bg)
shap_values = explainer.shap_values(X_exp)

# Multi-class SHAP handling
if isinstance(shap_values, list):
    # shap_values is a list of [N, F] arrays, one for each class
    # Convert to (Classes, Samples, Features)
    shap_arr = np.array(shap_values)
    # Calculate mean absolute SHAP across classes (axis 0) and samples (axis 1)
    mean_per_feature = np.abs(shap_arr).mean(axis=(0, 1))
    shap_to_plot = shap_values[0]
else:
    # Binary/Single output case
    mean_per_feature = np.abs(shap_values).mean(axis=0)
    shap_to_plot = shap_values

print("  Plotting SHAP beeswarm…")
plt.figure(figsize=(10, 9))
shap.summary_plot(
    shap_to_plot, X_exp.numpy(),
    feature_names=FEATURE_NAMES,
    max_display=20, show=False
)
plt.title(f'SHAP Beeswarm — Impact on {CLASS_NAMES[0]}\n(Top 20 Features)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('06_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

print("  Plotting SHAP global importance…")
# Ensure mean_per_feature is exactly length 40
feat_imp = pd.Series(mean_per_feature, index=FEATURE_NAMES).nlargest(20)

fig, ax = plt.subplots(figsize=(10, 8), facecolor=DARK_BG)
ax.set_facecolor(DARK_BG)
ax.spines[:].set_color(GRID_COL)
ax.tick_params(colors='#aaaaaa')

bar_colors2 = plt.cm.viridis(np.linspace(0.2, 0.85, len(feat_imp)))
feat_imp.sort_values().plot(kind='barh', ax=ax, color=bar_colors2, edgecolor='none')

ax.set_xlabel('Mean |SHAP Value| (Global Impact)', fontsize=11, color='#dddddd')
ax.set_title('Top 20 Most Important Features for IoT Attack Detection\n(Mean Absolute SHAP — All Classes)', fontsize=13, fontweight='bold', color='white')
ax.grid(axis='x', color=GRID_COL, lw=0.8)

plt.tight_layout()
plt.savefig('07_shap_bar.png', dpi=150, bbox_inches='tight', facecolor=DARK_BG)
plt.show()

global_model = global_model.to(DEVICE)
print("\n✅ SHAP explainability plots saved!")

In [ ]:
best_round_idx = history['test_acc'].index(max(history['test_acc']))

print("\n" + "═" * 68)
print("   📄 PAPER-READY RESULTS — FedProx IDS on CIC-IoT2023")
print("═" * 68)
print(f"  Dataset           : CIC-IoT2023")
print(f"  FL Algorithm      : FedProx (μ={MU})")
print(f"  Non-IID Partition : Dirichlet α={DIRICHLET_ALPHA}")
print(f"  # Clients         : {NUM_CLIENTS}")
print(f"  # FL Rounds       : {FL_ROUNDS}")
print(f"  Local Epochs/Rnd  : {LOCAL_EPOCHS}")
print(f"  # Classes         : {NUM_CLASSES}")
print(f"  # Features        : {len(FEATURE_NAMES)}")
print(f"  Model Parameters  : {sum(p.numel() for p in global_model.parameters()):,}")
print("─" * 68)
print(f"  Best Test Accuracy : {max(history['test_acc']):.4f}  "
      f"({max(history['test_acc']) * 100:.2f}%)")
print(f"  Best Test F1       : {max(history['test_f1']):.4f}")
print(f"  Final Precision    : {history['test_prec'][-1]:.4f}")
print(f"  Final Recall       : {history['test_rec'][-1]:.4f}")
print(f"  Macro AUC-ROC      : {macro_auc:.4f}")
print(f"  Best Round         : {best_round_idx + 1} / {FL_ROUNDS}")
print(f"  Total Training Time: {total_time / 60:.1f} minutes")
print("═" * 68)

# ── Per-class table ───────────────────────────────────────────────
per_class_df = pd.DataFrame({
    'Attack Class':  CLASS_NAMES,
    'Precision':     [round(report[c]['precision'], 4) for c in CLASS_NAMES],
    'Recall':        [round(report[c]['recall'],    4) for c in CLASS_NAMES],
    'F1-Score':      [round(report[c]['f1-score'],  4) for c in CLASS_NAMES],
    'Support':       [int(report[c]['support'])         for c in CLASS_NAMES],
    'AUC':           [round(roc_aucs.get(c, 0), 4)     for c in CLASS_NAMES],
})
print("\n📋 Per-Class Results:")
print(per_class_df.to_string(index=False))

# ── Save CSVs ─────────────────────────────────────────────────────
summary_df = pd.DataFrame({
    'Metric': ['Accuracy', 'F1 (Weighted)', 'Precision',
               'Recall', 'AUC (Macro)', 'Training Time (min)'],
    'Value':  [f'{max(history["test_acc"]):.4f}',
               f'{max(history["test_f1"]):.4f}',
               f'{history["test_prec"][-1]:.4f}',
               f'{history["test_rec"][-1]:.4f}',
               f'{macro_auc:.4f}',
               f'{total_time / 60:.1f}']
})
summary_df.to_csv('09_results_summary.csv', index=False)
per_class_df.to_csv('10_per_class_results.csv', index=False)

# ── Save convergence history ───────────────────────────────────────
pd.DataFrame(history).to_csv('11_fl_round_history.csv', index=False)

print("\n✅ All CSVs saved!")

In [ ]:
zip_name = 'FedShield_Results_CICIoT2023.zip'
all_files = [
    '01_non_iid_distribution.png',
    '02_fl_convergence.png',
    '03_confusion_matrix.png',
    '04_per_class_f1.png',
    '05_roc_curves.png',
    '06_shap_beeswarm.png',
    '07_shap_bar.png',
    '08_shap_per_class.png',
    '09_results_summary.csv',
    '10_per_class_results.csv',
    '11_fl_round_history.csv',
]

with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in all_files:
        if os.path.exists(f):
            zf.write(f)
            print(f"  ✔ Added: {f}")
        else:
            print(f"  ⚠ Skipped (not found): {f}")

colab_files.download(zip_name)
print(f"\n✅ Download started: {zip_name}")
print("\n" + "═" * 68)
print("   🎉 FED-SHIELD Complete! All outputs in the ZIP file.")
print("═" * 68)